# 🏁 Week 11 Take Home · Data Structures & Algorithms 1

**10 problems on Lessons 01–07, auto-graded.** This is where you prove to yourself that you *own* this week's
material. (Week 12 has its own take-home for Lessons 08–15.)

| | |
|---|---|
| ⏱ **Time needed** | 3–4 hours (spread it over a few sittings) |
| 📚 **Covers** | Lessons 01–07: arrays & two pointers, hashing, linked lists, stacks & queues, recursion, searching & binary search |
| 🧮 **Points** | 26 points for tests + 3 speed-bonus points = **29** |
| 📝 **Also** | a short written reflection on complexity (read by a facilitator) |

### How it works

1. **Run the setup cell first** (and again after restarting the kernel).
2. Each problem has a function **stub** that does `raise NotImplementedError`. Replace that line with your code.
3. Run the cell. Below your function, `run_tests(...)` checks it: `🎉 All tests passed!`, or exactly which
   case failed, what was expected and what you returned.
4. Three problems also have a **⚡ speed test** on a big input. It only runs once all normal tests pass.
   A correct but slow (for example O(n²)) solution gets the points but not the bonus.
5. At the end, run the **scoring cell** to see your score table.

### Rules

* **No solutions are in this notebook.** That's on purpose. Use your lesson notebooks and `DSA Cheat Sheet.md`.
* **Don't edit the `TESTS[...]` lists or the setup cell.** The facilitator re-runs everything with the original tests.
* You may add helper functions and extra cells. Standard library only (`collections`, `heapq`, `bisect`, …):
  no pandas/NumPy inside your solutions.
* Before each problem, follow **UMPIRE** (Lesson 00): Understand, Match, Plan, Implement, Review, Evaluate.
  Write the brute force first if you're stuck, then improve it.

Problems go roughly from easy to hard inside each topic. Difficulty: **Easy = 2 pts, Medium = 3 pts, Hard = 4 pts.**

## ⚙️ Setup (run me first)

This cell defines `ListNode` (for problem 5), the test runner, the speed tests and the grader.
You're welcome to read it, but please don't change it.

In [3]:
from dsa_kit import *

import copy
import heapq
import random
import time
from collections import Counter, defaultdict, deque

import pandas as pd


# ---------- helper structures some problems use ----------
class ListNode:
    """A singly linked list node (Lesson 04)."""
    def __init__(self, val, next=None):
        self.val, self.next = val, next

    def __repr__(self):
        return f"ListNode({self.val!r})"


def from_list(values):
    """[1, 2, 3] -> 1 -> 2 -> 3 -> None. Returns the head (or None)."""
    head = None
    for v in reversed(values):
        head = ListNode(v, head)
    return head


def to_list(head, limit=100_000):
    """Linked list -> Python list (stops after `limit` nodes in case of an accidental cycle)."""
    out = []
    while head is not None and len(out) < limit:
        out.append(head.val)
        head = head.next
    return out


# ---------- problem register: (id, function name, title, topic, level, points) ----------
META = [
    ("p01", "max_window_sum",      "Best k-day sales",            "Arrays & two pointers", "Easy",   2),
    ("p02", "pair_with_sum",       "Two prices that hit a total", "Arrays & two pointers", "Easy",   2),
    ("p03", "group_anagrams",      "Group anagrams",              "Hashing",               "Medium", 3),
    ("p04", "longest_consecutive", "Longest run of IDs",          "Hashing",               "Medium", 3),
    ("p05", "reverse_list",        "Reverse a linked list",       "Linked lists & stacks", "Easy",   2),
    ("p06", "is_balanced",         "Balanced brackets",           "Linked lists & stacks", "Easy",   2),
    ("p07", "daily_temperatures",  "Wait for a warmer day",       "Linked lists & stacks", "Medium", 3),
    ("p08", "flatten",             "Flatten a nested basket",     "Recursion",             "Easy",   2),
    ("p09", "search_range",        "First and last position",     "Searching",             "Medium", 3),
    ("p10", "ship_capacity",       "Smallest truck capacity",     "Searching",             "Hard",   4),
]
META_BY_ID = {m[0]: m for m in META}
TESTS = {}          # each problem cell fills TESTS["pXX"]

# Some functions take structures; adapters let the tests be written as plain lists.
ADAPT = {
    "p05": lambda fn: (lambda values: to_list(fn(from_list(values)))),
}


def _same_groups(got, expected):
    """Order-insensitive comparison for lists of groups."""
    return sorted(sorted(g) for g in got) == sorted(sorted(g) for g in expected)


COMPARE = {
    "p02": lambda got, valid: (got is None and not valid) or (got is not None and tuple(got) in valid),
    "p03": _same_groups,
    "p09": lambda got, expected: tuple(got) == tuple(expected),
}

# ---------- speed tests (+1 bonus point each): a correct but slow O(n^2) idea will not pass ----------
_EVENS = list(range(0, 8_000, 2))
_RUNS = [10 * block + j for block in range(2_000) for j in range(4)]
random.Random(1).shuffle(_RUNS)
_SORTED_BIG = [i // 3 for i in range(150_000)]
_QUERIES = list(range(0, 50_000, 167)) + [10**9]
SPEED = {   # id: (runner, expected, time limit in seconds)
    "p02": (lambda fn: fn(_EVENS, 15_990), {(3996, 3999), (3997, 3998)}, 0.05),
    "p04": (lambda fn: fn(_RUNS), 4, 0.05),
    "p09": (lambda fn: [tuple(fn(_SORTED_BIG, q)) for q in _QUERIES],
            [(3 * q, 3 * q + 2) for q in _QUERIES[:-1]] + [(-1, -1)], 0.05),
}


def speed_check(pid, fn, quiet=False):
    """Time fn on a big input. Returns True (fast and correct), False, or None (not implemented)."""
    if pid not in SPEED:
        return None
    runner, expected, limit = SPEED[pid]
    try:
        t0 = time.perf_counter()
        got = runner(fn)
        elapsed = time.perf_counter() - t0
    except NotImplementedError:
        return None
    except Exception as err:
        if not quiet:
            print(f"   ⚡ speed test crashed: {type(err).__name__}: {err}")
        return False
    correct = COMPARE.get(pid, lambda g, e: g == e)(got, expected)
    fast = elapsed <= limit
    if not quiet:
        if correct and fast:
            print(f"   ⚡ speed test passed: {elapsed * 1e3:.1f} ms (limit {limit * 1e3:.0f} ms). +1 bonus point")
        elif correct:
            print(f"   🐢 speed test: correct but {elapsed * 1e3:,.0f} ms (limit {limit * 1e3:.0f} ms). "
                  "Think about a better structure or algorithm.")
        else:
            print("   ⚡ speed test: wrong answer on the big input.")
    return correct and fast


def run_tests(pid, fn):
    """Run the tests for one problem (and its speed test, once all tests pass)."""
    target = ADAPT[pid](fn) if pid in ADAPT else fn
    ok = check(target, TESTS[pid], name=META_BY_ID[pid][1], compare=COMPARE.get(pid))
    if ok:
        speed_check(pid, fn)

print(f"Setup done: {len(META)} problems, {sum(m[5] for m in META)} points + {len(SPEED)} speed bonus points.")

Setup done: 10 problems, 26 points + 3 speed bonus points.


---
## A · Arrays & two pointers

### Problem 1

**Best k-day sales (Easy · 2 pts).** A Jumia seller has daily sales figures. Return the **largest total of any
k consecutive days**. If `k <= 0` or `k > len(nums)`, return `None`.

`max_window_sum([2, 1, 5, 1, 3, 2], 3)` → `9` (days 2–4: 5 + 1 + 3).

🎯 Target: **O(n)** time, **O(1)** extra space (sliding window, Lesson 02).

In [5]:
def max_window_sum(nums, k):
    """Largest sum of k consecutive values, or None if k is invalid."""
    if k <= 0 or k > len(nums):
        return None

    current_sum = sum(nums[:k])
    max_sum = current_sum

    for i in range(k, len(nums)):
        current_sum += nums[i] - nums[i - k]
        if current_sum > max_sum:
            max_sum = current_sum

    return max_sum


TESTS["p01"] = [
    (([2, 1, 5, 1, 3, 2], 3), 9),
    (([5], 1), 5),
    (([], 2), None),
    (([1, 2], 3), None),
    (([1, 2, 3], 0), None),
    (([-4, -2, -7], 2), -6),
    (([3, 3, 3, 3], 2), 6),
]
run_tests("p01", max_window_sum)

max_window_sum: 🎉 All tests passed!


### Problem 2

**Two prices that hit a total (Easy · 2 pts + ⚡1).** `sorted_nums` is sorted ascending. Return a pair of
indices `(i, j)` with `i < j` and `sorted_nums[i] + sorted_nums[j] == target`, or `None` if no pair exists.
If several pairs work, any one is accepted. You may not use the same index twice.

🎯 Target: **O(n)** time, **O(1)** space (two pointers, Lesson 02). ⚡ The speed test uses 4,000 numbers.

In [7]:
def pair_with_sum(sorted_nums, target):
    """Indices (i, j), i < j, of two values adding to target, or None."""
    left, right = 0, len(sorted_nums) - 1

    while left < right:
        current_sum = sorted_nums[left] + sorted_nums[right]
        if current_sum == target:
            return (left, right)
        elif current_sum < target:
            left += 1
        else:
            right -= 1

    return None


TESTS["p02"] = [   # expected = the set of ALL valid answers; returning any one of them passes
    (([1, 2, 4, 7, 11], 9), {(1, 3)}),
    (([1, 3, 5], 100), set()),
    (([], 5), set()),
    (([4], 8), set()),
    (([2, 2, 3], 4), {(0, 1)}),
    (([-5, -1, 0, 3, 6], 1), {(0, 4)}),
    (([1, 2, 3, 4, 5, 6], 7), {(0, 5), (1, 4), (2, 3)}),
]
run_tests("p02", pair_with_sum)

pair_with_sum: 🎉 All tests passed!
   ⚡ speed test passed: 2.1 ms (limit 50 ms). +1 bonus point


---
## B · Hashing

### Problem 3

**Group anagrams (Medium · 3 pts).** Two words are anagrams if they use exactly the same letters
(`"tea"` and `"eat"`). Group the words so that each group holds anagrams of each other. Return a list of groups.
The order of the groups, and the order inside each group, don't matter.

🎯 Target: **O(n · L log L)** for n words of length up to L. Hint: what could be the *dict key* of a group?

In [9]:
def group_anagrams(words):
    """List of groups (lists) of words that are anagrams of each other."""
    groups = defaultdict(list)
    for word in words:
        key = "".join(sorted(word))
        groups[key].append(word)
    return list(groups.values())


TESTS["p03"] = [
    (["eat", "tea", "tan", "ate", "nat", "bat"], [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]),
    ([], []),
    ([""], [[""]]),
    (["a"], [["a"]]),
    (["abc", "bca", "cab", "xyz"], [["abc", "bca", "cab"], ["xyz"]]),
    (["ab", "ba", "ab"], [["ab", "ba", "ab"]]),
    (["lagos", "goals", "kano", "oak"], [["lagos", "goals"], ["kano"], ["oak"]]),
]
run_tests("p03", group_anagrams)

group_anagrams: 🎉 All tests passed!


### Problem 4

**Longest run of IDs (Medium · 3 pts + ⚡1).** Given an **unsorted** list of integers (maybe with repeats),
return the length of the longest run of consecutive values. `[100, 4, 200, 1, 3, 2]` → `4` (1, 2, 3, 4).

🎯 Target: **O(n)** average with a set (sorting first gives O(n log n), which is also acceptable here).
⚠️ Checking `x + 1 in nums` on a *list* is O(n) per check. ⚡ The speed test uses 8,000 numbers.

In [11]:
def longest_consecutive(nums):
    """Length of the longest run of consecutive integers."""
    num_set = set(nums)
    max_streak = 0

    for num in num_set:
        # Check if `num` is the start of a sequence
        if num - 1 not in num_set:
            current_num = num
            current_streak = 1

            while current_num + 1 in num_set:
                current_num += 1
                current_streak += 1

            if current_streak > max_streak:
                max_streak = current_streak

    return max_streak


TESTS["p04"] = [
    ([100, 4, 200, 1, 3, 2], 4),
    ([], 0),
    ([7], 1),
    ([1, 2, 0, 1], 3),
    ([-3, -2, -1, 5], 3),
    ([10, 30, 20], 1),
    ([0, 3, 7, 2, 5, 8, 4, 6, 0, 1], 9),
]
run_tests("p04", longest_consecutive)

longest_consecutive: 🎉 All tests passed!
   ⚡ speed test passed: 7.2 ms (limit 50 ms). +1 bonus point


---
## C · Linked lists, stacks & queues

### Problem 5

**Reverse a linked list (Easy · 2 pts).** Given the `head` of a singly linked list built from `ListNode`s
(defined in the setup cell), reverse it **in place** and return the new head.
The tests turn plain lists into linked lists for you with `from_list`, and back with `to_list`.

🎯 Target: **O(n)** time, **O(1)** extra space (Lesson 04). Don't copy the values into a Python list.

In [13]:
def reverse_list(head):
    """Reverse a singly linked list in place; return the new head."""
    prev = None
    cur = head

    while cur is not None:
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt

    return prev


TESTS["p05"] = [
    ([1, 2, 3, 4, 5], [5, 4, 3, 2, 1]),
    ([], []),
    ([7], [7]),
    ([1, 2], [2, 1]),
    ([3, 3, 1], [1, 3, 3]),
]
run_tests("p05", reverse_list)

reverse_list: 🎉 All tests passed!


### Problem 6

**Balanced brackets (Easy · 2 pts).** Return `True` if every `(`, `[` and `{` in `text` is closed by the matching
bracket in the right order. Other characters are ignored. `"([]{})"` → `True`, `"([)]"` → `False`.

🎯 Target: **O(n)** time with a stack (Lesson 05).

In [15]:
def is_balanced(text):
    """True if all brackets in text are properly matched and nested."""
    stack = []
    pairs = {")": "(", "]": "[", "}": "{"}

    for char in text:
        if char in "([{":
            stack.append(char)
        elif char in pairs:
            if not stack or stack[-1] != pairs[char]:
                return False
            stack.pop()

    return len(stack) == 0


TESTS["p06"] = [
    ("()", True),
    ("([]{})", True),
    ("(]", False),
    ("([)]", False),
    ("", True),
    ("((", False),
    ("))", False),
    ("total = (price[0] + fee) * {rate}", True),
]
run_tests("p06", is_balanced)

is_balanced: 🎉 All tests passed!


### Problem 7

**Wait for a warmer day (Medium · 3 pts).** For each day in `temps`, how many days until a **strictly warmer**
day? Use 0 if there is none. `[73, 74, 75, 71, 69, 72, 76, 73]` → `[1, 1, 4, 2, 1, 1, 0, 0]`.

🎯 Target: **O(n)** with a monotonic stack (Lesson 05). The double loop is O(n²).

In [17]:
def daily_temperatures(temps):
    """answer[i] = days after day i until a warmer day (0 if never)."""
    n = len(temps)
    answer = [0] * n
    stack = []  # stores indices of temperatures

    for i, temp in enumerate(temps):
        while stack and temps[stack[-1]] < temp:
            prev_idx = stack.pop()
            answer[prev_idx] = i - prev_idx
        stack.append(i)

    return answer


TESTS["p07"] = [
    ([73, 74, 75, 71, 69, 72, 76, 73], [1, 1, 4, 2, 1, 1, 0, 0]),
    ([], []),
    ([30], [0]),
    ([30, 40, 50, 60], [1, 1, 1, 0]),
    ([60, 50, 40], [0, 0, 0]),
    ([30, 30, 31], [2, 1, 0]),
]
run_tests("p07", daily_temperatures)

daily_temperatures: 🎉 All tests passed!


---
## D · Recursion

### Problem 8

**Flatten a nested basket (Easy · 2 pts).** An order export arrives as **nested lists**: a basket can hold
items *and* smaller baskets, to any depth. Return a flat list of all the items, in left-to-right order.

`flatten([1, [2, [3, 4]], 5])` → `[1, 2, 3, 4, 5]`

🎯 Technique: **recursion** (Lesson 06). Base case: a non-list item. Recursive case: a list, so flatten each
element. (Use `isinstance(x, list)`.)

In [21]:
def flatten(nested):
    """All non-list items of a nested list, in left-to-right order."""
    result = []
    for item in nested:
        if isinstance(item, list):
            result.extend(flatten(item))
        else:
            result.append(item)
    return result


TESTS["p08"] = [
    (([1, [2, [3, 4]], 5],), [1, 2, 3, 4, 5]),
    (([],), []),
    (([[], [[]]],), []),
    (([7],), [7]),
    (([[[["deep"]]]],), ["deep"]),
    ((["rice", ["beans", ["oil", "salt"]], ["yam"]],), ["rice", "beans", "oil", "salt", "yam"]),
]
run_tests("p08", flatten)

flatten: 🎉 All tests passed!


---
## E · Searching & binary search

### Problem 9

**First and last position (Medium · 3 pts + ⚡1).** `nums` is sorted ascending and may contain repeats. Return
`(first_index, last_index)` of `target`, or `(-1, -1)` if it isn't there.

🎯 Target: **O(log n)** (two binary searches, Lesson 07). A linear scan is O(n).
⚡ The speed test runs 300 searches in a list of 150,000 numbers.

In [23]:
def search_range(nums, target):
    """(first, last) index of target in sorted nums, or (-1, -1)."""

    def find_boundary(is_first):
        lo, hi = 0, len(nums) - 1
        boundary = -1
        while lo <= hi:
            mid = (lo + hi) // 2
            if nums[mid] == target:
                boundary = mid
                if is_first:
                    hi = mid - 1
                else:
                    lo = mid + 1
            elif nums[mid] < target:
                lo = mid + 1
            else:
                hi = mid - 1
        return boundary

    first = find_boundary(is_first=True)
    if first == -1:
        return (-1, -1)
    last = find_boundary(is_first=False)
    return (first, last)


TESTS["p09"] = [
    (([5, 7, 7, 8, 8, 10], 8), (3, 4)),
    (([5, 7, 7, 8, 8, 10], 6), (-1, -1)),
    (([], 0), (-1, -1)),
    (([1], 1), (0, 0)),
    (([2, 2, 2, 2], 2), (0, 3)),
    (([1, 2, 3], 4), (-1, -1)),
    (([-3, -3, 0], -3), (0, 1)),
]
run_tests("p09", search_range)

search_range: 🎉 All tests passed!
   ⚡ speed test passed: 11.6 ms (limit 50 ms). +1 bonus point


### Problem 10

**Smallest truck capacity (Hard · 4 pts).** A logistics company must move parcels from Lagos to Ibadan in order
(`weights[0]` first, and so on) within `days` days. Each day the truck carries parcels in order until the next
one would go over its capacity. What is the **smallest capacity** that gets everything delivered in `days` days?

`ship_capacity([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5)` → `15`

🎯 Technique: **binary search on the answer** (Lesson 07). The answer lies between `max(weights)` and
`sum(weights)`, and "can we finish with capacity c?" is True for every c above some point. Target
O(n log(sum of weights)).

In [25]:
def ship_capacity(weights, days):
    """Smallest capacity that ships all weights, in order, within `days` days."""

    def can_ship(capacity):
        days_count = 1
        current_load = 0
        for w in weights:
            if current_load + w > capacity:
                days_count += 1
                current_load = 0
            current_load += w
        return days_count <= days

    lo = max(weights)
    hi = sum(weights)

    while lo < hi:
        mid = (lo + hi) // 2
        if can_ship(mid):
            hi = mid
        else:
            lo = mid + 1

    return lo


TESTS["p10"] = [
    (([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5), 15),
    (([3, 2, 2, 4, 1, 4], 3), 6),
    (([1, 2, 3, 1, 1], 4), 3),
    (([10], 1), 10),
    (([5, 5, 5], 3), 5),
    (([5, 5, 5], 1), 15),
    (([1, 1, 1, 1], 2), 2),
]
run_tests("p10", ship_capacity)

ship_capacity: 🎉 All tests passed!


---
## 📝 Written reflection (complexity)

Answer in the markdown cell below (double-click it to edit). Two or three sentences each is enough.
A facilitator reads this part: it shows you understand *why* your code is fast, not only *that* it passes.

1. **Problem 4 (longest run).** What are the time and space complexity of your solution? If you used a set,
   explain why the inner `while` loop does not make it O(n²). If you sorted, what did sorting cost you?
2. **Problem 7 (warmer day).** Your stack can hold many days. Why is the whole solution still O(n)?
   (Hint: how many times can each day be pushed and popped?)
3. **Problem 8 (flatten).** What is the base case and what is the recursive case? What would happen with a basket
   nested 5,000 levels deep, and how could you avoid it?
4. **Problem 9 (first and last position).** Why is binary search O(log n)? How many comparisons, roughly,
   for a list of 1,000,000 numbers?
5. **Problem 10 (truck capacity).** What exactly are you binary-searching over, and why is it allowed
   (what property must the yes/no question have)?
6. **Pick any problem where your first idea was too slow.** What was the brute force, what was its
   complexity, and which data structure or technique fixed it?

**My answers:**

1. Problem 4 (longest run): Time complexity is $O(n)$ average and space complexity is $O(n)$ for the set. The inner while loop only executes for values that are sequence starting points (num - 1 not in num_set), ensuring every number is visited at most twice across the entire execution. If sorted instead, sorting costs $O(n \log n)$ time complexity.
2. Problem 7 (warmer day): Every index is pushed onto the stack exactly once and popped at most once throughout the entire loop. Therefore, total push and pop operations are bounded by $2n$, yielding an amortized time complexity of $O(n)$.
3. Problem 8 (flatten): The base case is a non-list item (which is directly appended to the output list), and the recursive case is a list (where flatten is called on each element). A basket nested 5,000 levels deep would exceed Python's default stack recursion limit and throw a RecursionError; this could be avoided using an explicit stack/iterative approach or increasing sys.setrecursionlimit.
4. Problem 9 (first and last position): Binary search is $O(\log n)$ because it halves the search space at each iteration step ($n \to n/2 \to n/4 \dots$). For a list of 1,000,000 elements, $\log_2(1,000,000) \approx 20$ comparisons are needed per search.
5. Problem 10 (truck capacity): We binary-search over the candidate truck capacity range $[\max(\text{weights}), \sum(\text{weights})]$. This is valid because the predicate can_ship(capacity) is monotonic: if a capacity $C$ can ship all packages within $D$ days, any capacity $C' > C$ can also complete the task.
6. Problem Pick: For Problem 4, the initial brute-force approach was checking if $x + 1, x + 2, \dots$ existed using linear scans on a list, which resulted in $O(n^3)$ time complexity. Converting the array to a set provided $O(1)$ lookup time and filtering to sequence starters optimized the total execution to $O(n)$.

---
## 🧮 Score table

Run this cell when you're done (or any time, to see your progress). It quietly re-runs every problem's tests,
awards partial points for partially passing problems, adds the speed bonuses, and prints your total.

Grade bands: **Distinction ≥ 85% · Merit ≥ 70% · Pass ≥ 50%.**

In [28]:
def score_all():
    """Re-run every problem's tests quietly and print a score table."""
    rows = []
    for pid, name, title, topic, level, points in META:
        max_pts = points + (1 if pid in SPEED else 0)      # the speed bonus counts towards the maximum
        fn = globals().get(name)
        tests = TESTS.get(pid)
        if fn is None or tests is None:
            rows.append([pid, title, topic, level, "cell not run", "—", 0.0, max_pts])
            continue
        target = ADAPT[pid](fn) if pid in ADAPT else fn
        compare = COMPARE.get(pid, lambda got, exp: got == exp)
        passed, attempted = 0, True
        for args, expected in tests:
            call_args = args if isinstance(args, tuple) else (args,)
            try:
                ok = compare(target(*copy.deepcopy(call_args)), expected)
            except NotImplementedError:
                attempted = False
                break
            except Exception:
                ok = False
            passed += bool(ok)
        if not attempted:
            rows.append([pid, title, topic, level, "not attempted", "—", 0.0, max_pts])
            continue
        earned = points * passed / len(tests)
        speed = "—"
        if pid in SPEED:
            speed = "not yet"
            if passed == len(tests):
                fast = speed_check(pid, fn, quiet=True)
                speed = "✅ +1" if fast else "🐢 slow"
                earned += 1 if fast else 0
        rows.append([pid, title, topic, level, f"{passed}/{len(tests)}", speed, round(earned, 2), max_pts])
    table = pd.DataFrame(rows, columns=["#", "Problem", "Topic", "Level", "Tests passed", "Speed bonus",
                                        "Points", "Max"])
    max_points = sum(m[5] for m in META) + len(SPEED)
    total = table["Points"].sum()
    pct = 100 * total / max_points
    band = ("Distinction" if pct >= 85 else "Merit" if pct >= 70 else "Pass" if pct >= 50 else "Not yet: keep going!")
    print(table.to_string(index=False))
    print(f"\nTOTAL: {total:.1f} / {max_points}  ({pct:.0f}%)  ->  {band}")
    by_topic = table.groupby("Topic", sort=False)[["Points", "Max"]].sum()
    print("\nBy topic:")
    print(by_topic.to_string())
    return total, max_points

TOTAL, MAX_POINTS = score_all()

  #                     Problem                 Topic  Level Tests passed Speed bonus  Points  Max
p01            Best k-day sales Arrays & two pointers   Easy          7/7           —     2.0    2
p02 Two prices that hit a total Arrays & two pointers   Easy          7/7        ✅ +1     3.0    3
p03              Group anagrams               Hashing Medium          7/7           —     3.0    3
p04          Longest run of IDs               Hashing Medium          7/7        ✅ +1     4.0    4
p05       Reverse a linked list Linked lists & stacks   Easy          5/5           —     2.0    2
p06           Balanced brackets Linked lists & stacks   Easy          8/8           —     2.0    2
p07       Wait for a warmer day Linked lists & stacks Medium          6/6           —     3.0    3
p08     Flatten a nested basket             Recursion   Easy          6/6           —     2.0    2
p09     First and last position             Searching Medium          7/7        ✅ +1     4.0    4
p10     Sm

### Submitting

1. **Kernel → Restart & Run All.** Make sure every cell runs without errors from a fresh start.
2. Check the score table and that your reflection is filled in.
3. Save the notebook and submit it as instructed by your facilitator (file name:
   `Take Home - <your name>.ipynb`).

Good luck. You've got this. 💪

<!-- practice:start -->
---
## 🔁 SQL Practice

The DSA ideas from this week live inside every database too. `GROUP BY` is a hash map, `EXISTS` is a search that stops at the first hit, and window functions give you sliding windows and prefix sums. These SQL tasks are **extra practice**: they are not part of the graded score table above.

**How it works**
1. Run the **practice setup** cell first (and again after restarting the kernel). It connects with your **read-only** account.
2. **SQL tasks:** write your query between the triple quotes and run the cell. `p.sql(...)` shows your result and
   says ✅ or ❌, and whether the **shape** (rows × columns) is right. Match the column order, sort order and rounding
   asked for (column *names* aren't checked).
3. Run `p.score()` at the end to see your practice score.

Refresh your memory with `Week-6 (Numpy & Pandas-1)/4-Intermediate SQL/Intermediate SQL.ipynb`.

In [1]:
# --- practice setup: run this cell first (don't edit it) ---
import sys, pathlib
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.practice import Practice, q

EXPECTED = {
 "s01": [
  [
   10,
   2
  ],
  "c137041ce374ba0eebae1a0dc26f47624e10c4409a790f67fb0c69151d1d9e0c"
 ],
 "s02": [
  [
   6,
   3
  ],
  "707fcfb01aeaa5808b01eecbfcef5d31b534cad4fb0000bc4b894f07dd7840a6"
 ],
 "s03": [
  [
   12,
   4
  ],
  "560a169ec6245f6c721b18efb8baee2c4cbe2b4530547395ce2f59ef1054ecbb"
 ],
 "s04": [
  [
   6,
   3
  ],
  "12501409aa327467847132eea2feafcd4bbe7ce6376e6f07f3290f4fc01030fa"
 ]
}   # fingerprints only: the answers are not stored here
TITLES = {
 "s01": "Hashing: count values like a Counter",
 "s02": "Searching: new customers with NOT EXISTS",
 "s03": "Prefix sums: running totals and range sums",
 "s04": "Sliding window: each city's warmest week"
}
p = Practice(EXPECTED, TITLES)
print(f"Practice ready: {len(TITLES)} tasks. Connected as:", q("SELECT CURRENT_USER AS u").u[0])

Practice ready: 4 tasks. Connected as: intern_readonly


### SQL Task 1 · Hashing: count values like a Counter

In Lesson 03 you counted things with a `dict` / `Counter`. In SQL, `GROUP BY` does the same job: the database puts
every row in a "bucket" by its key and counts each bucket.

Using London's 2014 weather (`week06.london_2014`), count how many days had each **maximum temperature**, and keep
only the **common** values: those that happened on **at least 20 days**.

**Return:** `max_temp_c`, `days`. Sort by `days` (most first), then `max_temp_c` (highest first) for ties.
💡 `GROUP BY [Max TemperatureC]` and `HAVING COUNT(*) >= 20`. This is
`Counter(temps).most_common()` plus a filter.

In [3]:
SQL_01 = """
SELECT 
    [Max TemperatureC] AS max_temp_c,
    COUNT(*) AS days
FROM week06.london_2014
GROUP BY [Max TemperatureC]
HAVING COUNT(*) >= 20
ORDER BY days DESC, max_temp_c DESC;
"""

p.sql("s01", SQL_01)

,max_temp_c,days
0,18,28
1,19,27
2,14,27
3,11,26
4,21,24
5,22,23
6,12,22
7,9,22
8,16,21
9,13,21


✅ s01: correct!


,max_temp_c,days
0,18,28
1,19,27
2,14,27
3,11,26
4,21,24
5,22,23
6,12,22
7,9,22
8,16,21
9,13,21


### SQL Task 2 · Searching: new customers with NOT EXISTS

Which countries bought milk from the UK in **January–May 2015** but **never in 2014**? These are new customers.

- 2015 rows: `week09.comtrade_milk_uk_jan_jul_15`, with `[Trade Flow] = 'Exports'`.
- 2014 rows: `week09.countrymilk` (also `'Exports'`).

**Return:** `partner`, `exports_kusd` (their 2015 exports in **thousands** of US$, **rounded to 1 dp**),
`months` (how many different months of 2015 they bought in). Sort by `exports_kusd` (highest first), then `partner`.
⚠️ The 2015 table has `Partner = 'World'` total rows: leave them out.
💡 `WHERE NOT EXISTS (SELECT 1 FROM week09.countrymilk c WHERE c.Partner = e.Partner AND ...)`. Like a search that
returns as soon as it finds one match, `EXISTS` can stop at the first matching row. A `NOT IN` on a column that
might contain NULLs can silently return nothing, so `NOT EXISTS` is the safer habit.

🔎 **Look closely at your result:** one "partner" is `Areas, nes`. That's a trade-statistics code meaning *not elsewhere specified* (trade that couldn't be assigned to a country), **not** a real country. Keep it in your answer, but in a real report you'd flag or exclude it, just like the `'World'` row.

In [5]:
SQL_02 = """
SELECT 
    e.Partner AS partner,
    ROUND(SUM(e.[Trade Value (US$)]) / 1000.0, 1) AS exports_kusd,
    COUNT(DISTINCT e.Period) AS months
FROM week09.comtrade_milk_uk_jan_jul_15 e
WHERE e.[Trade Flow] = 'Exports'
  AND e.Partner <> 'World'
  AND NOT EXISTS (
      SELECT 1 
      FROM week09.countrymilk c 
      WHERE c.[Trade Flow] = 'Exports' 
        AND c.Partner = e.Partner
  )
GROUP BY e.Partner
ORDER BY exports_kusd DESC, partner ASC;
"""

p.sql("s02", SQL_02)

,partner,exports_kusd,months
0,"Areas, nes",51.2,5
1,Togo,49.1,1
2,Zambia,48.2,1
3,Uruguay,46.0,1
4,Turkey,37.9,1
5,Ethiopia,2.2,1


✅ s02: correct!


,partner,exports_kusd,months
0,"Areas, nes",51.2,5
1,Togo,49.1,1
2,Zambia,48.2,1
3,Uruguay,46.0,1
4,Turkey,37.9,1
5,Ethiopia,2.2,1


### SQL Task 3 · Prefix sums: running totals and range sums

Prefix sums answer "what is the total from month *a* to month *b*?" with one subtraction:
`sum(a..b) = prefix[b] − prefix[a − 1]`. Build them for the UK's **2014 milk exports** (`week09.countrymilk`).

**Return one row per month:**
- `period` (e.g. `201401`)
- `exports_musd`: that month's exports
- `running_total_musd`: exports from January up to and including this month (the prefix sum)
- `last_3m_musd`: exports of this month and the 2 months before it, computed **by differencing**:
  running total now − running total 3 months earlier. Use `NULL` for January and February (they don't have 3
  months yet). For March it equals the running total.

All money in **millions of US$, rounded to 2 dp**; round only at the end. Sort by `period`.
💡 `SUM(v) OVER (ORDER BY period ROWS UNBOUNDED PRECEDING)` in a CTE, then
`LAG(running, 3, 0) OVER (ORDER BY period)` for "3 months earlier" (the `0` stands in for "before January").
Check: your March, June, September and December values should match the quarter totals from Week 8.

In [7]:
SQL_03 = """
WITH MonthlyExports AS (
    SELECT 
        Period AS period,
        SUM([Trade Value (US$)]) / 1000000.0 AS exports_musd
    FROM week09.countrymilk
    WHERE [Trade Flow] = 'Exports'
    GROUP BY Period
),
RunningTotals AS (
    SELECT 
        period,
        exports_musd,
        SUM(exports_musd) OVER (ORDER BY period ROWS UNBOUNDED PRECEDING) AS running_total_musd
    FROM MonthlyExports
)
SELECT 
    period,
    ROUND(exports_musd, 2) AS exports_musd,
    ROUND(running_total_musd, 2) AS running_total_musd,
    CASE 
        WHEN ROW_NUMBER() OVER (ORDER BY period) < 3 THEN NULL
        ELSE ROUND(running_total_musd - LAG(running_total_musd, 3, 0) OVER (ORDER BY period), 2)
    END AS last_3m_musd
FROM RunningTotals
ORDER BY period ASC;
"""

p.sql("s03", SQL_03)

,period,exports_musd,running_total_musd,last_3m_musd
0,201401,87.14,87.14,NaN
1,201402,72.49,159.63,NaN
2,201403,86.78,246.41,246.41
3,201404,95.20,341.61,254.47
4,201405,96.20,437.81,278.18
5,201406,99.32,537.12,290.71
6,201407,72.88,610.00,268.40
7,201408,61.74,671.74,233.93
8,201409,60.60,732.34,195.22
9,201410,57.00,789.34,179.34


✅ s03: correct!


,period,exports_musd,running_total_musd,last_3m_musd
0,201401,87.14,87.14,NaN
1,201402,72.49,159.63,NaN
2,201403,86.78,246.41,246.41
3,201404,95.20,341.61,254.47
4,201405,96.20,437.81,278.18
5,201406,99.32,537.12,290.71
6,201407,72.88,610.00,268.40
7,201408,61.74,671.74,233.93
8,201409,60.60,732.34,195.22
9,201410,57.00,789.34,179.34


### SQL Task 4 · Sliding window: each city's warmest week

Problem 1 found the best *k* consecutive days with a sliding window. Do it in SQL for all **six cities** in
`week06` (Beijing, Brasilia, Cape Town, Delhi, London, Moscow): find each city's **warmest 7-day period** in 2014,
by the **average of `Mean TemperatureC`** over the 7 days. A period is named by its **start day** and must contain
7 full days. If two periods tie, take the earliest.

**Return:** `city` (spelled exactly as in the list above), `start_day` (a date), `avg_mean_temp_c` (**rounded to
1 dp**). Sort by `city`.
⚠️ London's date column is `GMT`; the other five tables use `[Date]`. Both are **text** like `'2014-7-17'`, so convert
with `TRY_CONVERT(date, ...)` before you sort, or July will come after December.
⚠️ `Mean TemperatureC` is an integer: use `1.0 *` (or `AVG(1.0 * t)`) so the average isn't cut to a whole number.
💡 `UNION ALL` the six cities into one CTE with a `city` label. Then
`AVG(1.0 * t) OVER (PARTITION BY city ORDER BY day ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING)` plus a `COUNT(*)` over
the same frame (keep only windows with 7 days), and `ROW_NUMBER()` to pick each city's best.

In [15]:
SQL_04 = """
WITH CombinedWeather AS (
    SELECT 'Beijing' AS city, TRY_CONVERT(date, [Date]) AS day, 1.0 * [Mean TemperatureC] AS temp FROM week06.beijing_pek_2014
    UNION ALL
    SELECT 'Brasilia' AS city, TRY_CONVERT(date, [Date]) AS day, 1.0 * [Mean TemperatureC] AS temp FROM week06.brasilia_bsb_2014
    UNION ALL
    SELECT 'Cape Town' AS city, TRY_CONVERT(date, [Date]) AS day, 1.0 * [Mean TemperatureC] AS temp FROM week06.capetown_cpt_2014
    UNION ALL
    SELECT 'Delhi' AS city, TRY_CONVERT(date, [Date]) AS day, 1.0 * [Mean TemperatureC] AS temp FROM week06.delhi_del_2014
    UNION ALL
    SELECT 'London' AS city, TRY_CONVERT(date, [GMT]) AS day, 1.0 * [Mean TemperatureC] AS temp FROM week06.london_2014
    UNION ALL
    SELECT 'Moscow' AS city, TRY_CONVERT(date, [Date]) AS day, 1.0 * [Mean TemperatureC] AS temp FROM week06.moscow_svo_2014
),
WindowStats AS (
    SELECT
        city,
        day AS start_day,
        AVG(temp) OVER (
            PARTITION BY city
            ORDER BY day
            ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
        ) AS avg_temp,
        COUNT(*) OVER (
            PARTITION BY city
            ORDER BY day
            ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
        ) AS window_count
    FROM CombinedWeather
),
RankedWindows AS (
    SELECT
        city,
        start_day,
        ROUND(avg_temp, 1) AS avg_mean_temp_c,
        ROW_NUMBER() OVER (
            PARTITION BY city
            ORDER BY avg_temp DESC, start_day ASC
        ) AS rnk
    FROM WindowStats
    WHERE window_count = 7
)
SELECT
    city,
    start_day,
    avg_mean_temp_c
FROM RankedWindows
WHERE rnk = 1
ORDER BY city ASC;
"""

p.sql("s04", SQL_04)

,city,start_day,avg_mean_temp_c
0,Beijing,2014-07-17,29.0
1,Brasilia,2014-10-14,27.1
2,Cape Town,2014-02-12,25.9
3,Delhi,2014-08-15,28.0
4,London,2014-07-15,22.3
5,Moscow,2014-07-28,24.3


✅ s04: correct!


,city,start_day,avg_mean_temp_c
0,Beijing,2014-07-17,29.0
1,Brasilia,2014-10-14,27.1
2,Cape Town,2014-02-12,25.9
3,Delhi,2014-08-15,28.0
4,London,2014-07-15,22.3
5,Moscow,2014-07-28,24.3


### Practice score
Run this when you're done.

In [17]:
p.score()

Task Type                                      Topic Result
 s01  SQL       Hashing: count values like a Counter      ✅
 s02  SQL   Searching: new customers with NOT EXISTS      ✅
 s03  SQL Prefix sums: running totals and range sums      ✅
 s04  SQL   Sliding window: each city's warmest week      ✅
SQL score: 4 / 4


,Task,Type,Topic,Result
0,s01,SQL,Hashing: count values like a Counter,✅
1,s02,SQL,Searching: new customers with NOT EXISTS,✅
2,s03,SQL,Prefix sums: running totals and range sums,✅
3,s04,SQL,Sliding window: each city's warmest week,✅


<!-- practice:end -->

---
## 📰 Article Link

Write a **detailed article** about this week's work. It should cover:

- **What you learnt:** the key concepts, tools and techniques from this week.
- **What you solved:** the tasks and projects you worked on, and how you approached them.
- **Challenges you faced:** what was hard, confusing or didn't work at first.
- **How you overcame them:** what you tried, what finally worked, and what you'd do differently.
- **Anything else you want to share:** insights, results, visuals, or advice for others learning this.

Publish it, then paste the article's **URL** in the empty cell below (double-click the cell to edit it).